# YZM212 Makine Ogrenmesi 4. Lab Odevi: Uzak Bir Galaksinin Parlaklik Analizi
Bayesian Inference ve `emcee` kutuphanesi kullanarak parametre kestirimi uygulamasi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import corner

### 1. Veri Olusturma (Sentetik Gozlem)
Doga tarafindan bilinen gercek degerlerle sentetik bir dataset hazirliyoruz.

In [ ]:
true_mu = 150.0 # Gercek parlaklik
true_sigma = 10.0 # Gozlem hatasi
n_obs = 50 # Gozlem sayisi

np.random.seed(42)
data = true_mu + true_sigma * np.random.randn(n_obs)

print(f'Ornek Veriler: {data[:5]}')

### 2. Bayesyen Fonksiyonlarin Tanimlanmasi
Log-Likelihood (Olabilirlik), Log-Prior (On Bilgi) ve ikisini toplayan Log-Probability (Posterior) fonksiyonlari:

In [ ]:
def log_likelihood(theta, data):
    mu, sigma = theta
    if sigma <= 0: return -np.inf
    return -0.5 * np.sum(((data - mu) / sigma)**2 + np.log(2 * np.pi * sigma**2))

def log_prior(theta):
    mu, sigma = theta
    if 0 < mu < 300 and 0 < sigma < 50:
        return 0.0
    return -np.inf

def log_probability(theta, data):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, data)

### 3. MCMC Ornekleyiciyi Calistirma
32 adet Walker (gezgin) kullanarak her biri 2000 adim atacak sekilde ayarliyoruz. Ilk 500 adimi (burn-in) silecegiz.

In [ ]:
initial = [140, 5]
n_walkers = 32
np.random.seed(42)
pos = initial + 1e-4 * np.random.randn(n_walkers, 2)

sampler = emcee.EnsembleSampler(n_walkers, 2, log_probability, args=(data,))
sampler.run_mcmc(pos, 2000, progress=True)

flat_samples = sampler.get_chain(discard=500, thin=15, flat=True)

### 4. Sonuclarin Gorsellestirilmesi (Corner Plot)

In [ ]:
fig = corner.corner(
    flat_samples, labels=[r"$\mu$ (Parlaklik)", r"$\sigma$ (Hata)"],
    truths=[true_mu, true_sigma]
)
plt.show()

### 5. Analiz ve Tablo Degerleri

| Degisken | Gercek Deger (Girdi) | Tahmin Edilen (Median) | Alt Sinir (%16) | Ust Sinir (%84) | Mutlak Hata |
| :--- | :--- | :--- | :--- | :--- | :--- |
| $\mu$ (Parlaklik) | 150.0 | 147.78 | 146.39 | 149.12 | 2.22 |
| $\sigma$ (Hata Payi) | 10.0 | 9.47 | 8.63 | 10.58 | 0.53 |